# 08 — CatBoost-only diversity ensemble
The 04/06 soft-vote failed because RF/ET/GB are individually weak (0.79–0.81) and dragged the average down. A useful ensemble needs members that are **both strong and decorrelated**. Here every member is a strong CatBoost (~0.82–0.83), made diverse *structurally*:
- depth ∈ {5, 6, 7, 8} (different bias/variance trade-offs),
- one `MultiClassOneVsAll` head (decouples the per-class gradient from the softmax),
- one `rsm=0.7` column-subsampled model (random-subspace decorrelation).

Combine by **equal-weight** probability averaging — *no* OOF weight search (it collapsed to CatBoost=1.0 before and would just overfit the ±0.0017 noise floor measured in 07). We judge the ensemble strictly against the repeated-CV **gate**: it is a real gain only if it beats the single deployed model on the same folds by more than ~1σ.

In [1]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [2]:
def log_result(step, model, feature_set, oof_macro, pcf, notes=""):
    """Write one row to results_log.csv. Idempotent per (step, model): re-running
    a notebook replaces its own row rather than duplicating it."""
    import csv
    path = HERE / "results_log.csv"
    header = ["step", "model", "feature_set", "oof_macro_f1",
              "f1_Wake", "f1_Light", "f1_Deep", "f1_REM", "notes"]
    rows = []
    if path.exists():
        with open(path, newline="") as fh:
            data = list(csv.reader(fh))
        if data and data[0] == header:
            rows = [r for r in data[1:] if not (len(r) >= 2 and r[0] == step and r[1] == model)]
    row = [step, model, feature_set, round(float(oof_macro), 5),
           pcf["Wake"], pcf["Light"], pcf["Deep"], pcf["REM"], notes]
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header); w.writerows(rows); w.writerow(row)
    print("logged:", step, model, "OOF macro-F1 =", round(float(oof_macro), 5))


In [3]:
from catboost import CatBoostClassifier
import itertools
cols = json.load(open(ART / "feature_cols.json"))["columns"]
Xtr = pd.DataFrame(np.load(ART / "features_v1_train.npy"), columns=cols)
Xte = pd.DataFrame(np.load(ART / "features_v1_test.npy"), columns=cols)
y = np.load(ART / "y_train.npy")
test_ids = np.load(ART / "test_ids.npy")
folds = load_folds()
gate = json.load(open(ART / "repeated_cv_gate.json"))
print("features_v1:", Xtr.shape[1], "cols | gate:", gate["mean"], "+/-", gate["std"])

def norm_rows(p):
    """Row-normalize to sum 1 (MultiClassOneVsAll probs need it; no-op for softmax)."""
    s = p.sum(1, keepdims=True)
    return p / np.where(s == 0, 1.0, s)

def make_member(depth=6, loss="MultiClass", rsm=None, seed=SEED):
    kw = dict(loss_function=loss, eval_metric="TotalF1:average=Macro",
        iterations=3000, learning_rate=0.03, depth=depth, l2_leaf_reg=3.0,
        random_seed=seed, od_type="Iter", od_wait=150, use_best_model=True,
        thread_count=-1, allow_writing_files=False, verbose=False)
    if rsm is not None:
        kw["rsm"] = rsm
    return CatBoostClassifier(**kw)

features_v1: 28 cols | gate: 0.82598 +/- 0.00169


## Train the diverse members (same folds, OOF harness)
Each member runs through the shared `run_oof` on the saved folds, so all OOF/test matrices are row-aligned and averageable. The `cat_d6` member is the exact deployed config — our anchor for the gate comparison.

In [4]:
MEMBERS = [
    ("cat_d6",  dict(depth=6, loss="MultiClass")),                 # anchor (= sub01 config)
    ("cat_d5",  dict(depth=5, loss="MultiClass")),
    ("cat_d7",  dict(depth=7, loss="MultiClass")),
    ("cat_d8",  dict(depth=8, loss="MultiClass")),
    ("cat_ova", dict(depth=6, loss="MultiClassOneVsAll")),         # one-vs-all head
    ("cat_rsm", dict(depth=6, loss="MultiClass", rsm=0.7, seed=123)),  # column subsampling
]
m_oof, m_test, m_macro = {}, {}, {}
for name, cfg in MEMBERS:
    oof, tst, mac, _ = run_oof(lambda c=cfg: make_member(**c), Xtr, y, Xte, folds,
                               needs_impute=False, use_eval_set=True)
    m_oof[name], m_test[name] = norm_rows(oof), norm_rows(tst)
    m_macro[name] = mac
    print(f"  {name:<8} {str(cfg):<55} OOF macro-F1 = {mac:.5f}")

  cat_d6   {'depth': 6, 'loss': 'MultiClass'}                      OOF macro-F1 = 0.82898


  cat_d5   {'depth': 5, 'loss': 'MultiClass'}                      OOF macro-F1 = 0.82152


  cat_d7   {'depth': 7, 'loss': 'MultiClass'}                      OOF macro-F1 = 0.82678


  cat_d8   {'depth': 8, 'loss': 'MultiClass'}                      OOF macro-F1 = 0.82573


  cat_ova  {'depth': 6, 'loss': 'MultiClassOneVsAll'}              OOF macro-F1 = 0.82273


  cat_rsm  {'depth': 6, 'loss': 'MultiClass', 'rsm': 0.7, 'seed': 123} OOF macro-F1 = 0.82466


## Equal-weight ensemble + diversity check
Simple mean of the member probabilities. We also report mean pairwise prediction **disagreement** — the ensemble can only help to the extent members actually disagree.

In [5]:
names = [n for n, _ in MEMBERS]
ens_oof = sum(m_oof[n] for n in names) / len(names)
ens_test = sum(m_test[n] for n in names) / len(names)
ens_macro = macro_f1(y, ens_oof.argmax(1))

preds = {n: m_oof[n].argmax(1) for n in names}
dis = [float((preds[a] != preds[b]).mean()) for a, b in itertools.combinations(names, 2)]
anchor = m_macro["cat_d6"]                     # single deployed model, same folds
print("member OOF macro-F1:", {n: round(m_macro[n], 5) for n in names})
print("mean pairwise disagreement:", round(float(np.mean(dis)), 4))
print(f"\nENSEMBLE OOF macro-F1 = {ens_macro:.5f}")
print(f"anchor (single cat_d6) = {anchor:.5f}   delta = {ens_macro - anchor:+.5f}")
print(f"gate noise floor (1 sigma) = {gate['std']:.5f}")
real = ens_macro > anchor + gate["std"]
print("=> real gain beyond noise?", bool(real),
      "(need >", round(anchor + gate["std"], 5), ")")
print("ensemble per-class F1:", per_class_f1(y, ens_oof.argmax(1)))
np.save(ART / "cat_ensemble_oof.npy", ens_oof)
np.save(ART / "cat_ensemble_test.npy", ens_test)
json.dump({"members": names, "member_macro": {n: round(m_macro[n], 5) for n in names},
           "ensemble_macro": round(float(ens_macro), 5), "anchor": round(float(anchor), 5),
           "mean_disagreement": round(float(np.mean(dis)), 4),
           "beats_gate": bool(real)}, open(ART / "cat_ensemble.json", "w"), indent=2)
log_result("08_ensemble", "catboost_diversity_ensemble", "features_v1", ens_macro,
           per_class_f1(y, ens_oof.argmax(1)),
           f"equal-wt {len(names)} CatBoosts; vs anchor {anchor:.5f} ({ens_macro-anchor:+.5f}); disag={np.mean(dis):.3f}")

member OOF macro-F1: {'cat_d6': 0.82898, 'cat_d5': 0.82152, 'cat_d7': 0.82678, 'cat_d8': 0.82573, 'cat_ova': 0.82273, 'cat_rsm': 0.82466}
mean pairwise disagreement: 0.037

ENSEMBLE OOF macro-F1 = 0.82324
anchor (single cat_d6) = 0.82898   delta = -0.00574
gate noise floor (1 sigma) = 0.00169
=> real gain beyond noise? False (need > 0.83067 )
ensemble per-class F1: {'Wake': 0.8495, 'Light': 0.8469, 'Deep': 0.7694, 'REM': 0.8271}
logged: 08_ensemble catboost_diversity_ensemble OOF macro-F1 = 0.82324


## Submission (`sub04`) — diverse CatBoost-family hedge
Written as a candidate private-LB safety submission (single algorithm family, clean to defend). Whether it becomes a *final* pick depends on the gate verdict above — if it does not beat the anchor by >1σ it is a diversity hedge, not an improvement.

In [6]:
name4 = "sub04_CatBoostDiversityEnsemble_GBDT.csv"
pred4 = ens_test.argmax(1).astype(int)
sub4 = pd.DataFrame({"id": test_ids, "sleep_stage": pred4})
assert sub4.shape == (5000, 2)
assert sub4["id"].tolist() == list(range(9000, 14000))
assert sub4["sleep_stage"].isin(CLASSES).all()
sub4.to_csv(SUB / name4, index=False)
print("wrote", SUB / name4, "| OOF macro-F1:", round(ens_macro, 5))
print("class counts:", sub4["sleep_stage"].value_counts().sort_index().to_dict())
single_test = np.load(ART / "catboost_tuned_test.npy")
diff = int((ens_test.argmax(1) != single_test.argmax(1)).sum())
print(f"ensemble vs single-CatBoost (sub01) test preds differ on {diff} of 5000 rows")

wrote C:\Users\rasul\Documents\workspace_ws\yessenov_data_lab_program\day9\submissions\sub04_CatBoostDiversityEnsemble_GBDT.csv | OOF macro-F1: 0.82324
class counts: {0: 1114, 1: 1308, 2: 1270, 3: 1308}
ensemble vs single-CatBoost (sub01) test preds differ on 77 of 5000 rows


### Takeaways
- Members are all CatBoost on the same features, so they stay fairly correlated; equal-weight averaging smooths variance but, near this synthetic ceiling, is expected to land **within the ±0.0017 gate noise** of the single model — an honest result, not a failure.
- Kept as a single-algorithm-family **diversity hedge** (`sub04`). The single CatBoost (`sub01`) and the seed-bag (`sub03`) remain the primary private-LB pair unless `sub04` beats the gate.